# Simple baseline (Elo)

This file trains a simple logistic regression baseline using sklearn, to see how predictive raw elo is in terms of rating, 
not accounting for any board position. It's very possible that we achieve a high baseline even with raw elo. Understanding what that baseline is will prevent us from prematurely declaring victory when it's not much higher due to all the boards we feed in.

In [145]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [146]:
# Recap of what our data looks like

data_path = "subset.parquet"

df = pd.read_parquet(data_path)
df.head()

,lichess_id,tournament,movedata,clocks_white,clocks_black,evals,ply_count,white,black,white_rating,...,time_increment,result,termination,white_rating_diff,black_rating_diff,eco,opening,white_title,black_title,utc_timestamp
0,eh5dKZuD,NaN,b'\xf1\x12\xef\xae\x9c\x86\xe3\x01\xe50\xfe\x9...,"[180, 179, 177, 176, 175, 171, 165, 162, 158, ...","[180, 179, 178, 175, 173, 163, 154, 144, 140, ...",None,29,Panchito0O,PauloPeru78,1247,...,0.0,1-0,Normal,5.0,-5.0,C25,Vienna Game: Anderssen Defense,NaN,NaN,2025-01-01 00:00:05
1,6yXyLl2M,NaN,b'e\x9f\xe3\xbf\xa9\xdc>\x8c\xea[G\x00_k:\xf5)...,"[180, 179, 178, 177, 174, 173, 172, 171, 169, ...","[180, 180, 179, 178, 178, 177, 176, 176, 174, ...",None,104,igloknight,atacan3131,1577,...,0.0,0-1,Time forfeit,-6.0,6.0,B12,Caro-Kann Defense: Masi Variation,NaN,NaN,2025-01-01 00:00:05
2,SZW2amZz,NaN,"b""-\xb4K\xb2C\xdf\xa3Wem\x88(\xfe'|\x93\xa2Q`\...","[180, 179, 179, 177, 175, 173, 172, 171, 169, ...","[180, 178, 175, 174, 173, 171, 169, 168, 163, ...",None,87,draughts2chess,xhoxhi64,1043,...,0.0,1-0,Normal,10.0,-5.0,D00,Queen's Pawn Game,NaN,NaN,2025-01-01 00:00:05
3,VGhr7swl,NaN,b'\xb3s\xaff\x0b\x8c|\x94\xf7\xcb\xbb\xa7\xb22...,"[180, 176, 174, 172, 168, 167, 165, 158, 142, ...","[180, 180, 180, 179, 177, 176, 176, 166, 158, ...",None,58,GodofPastries,Mickey187,2015,...,0.0,1-0,Normal,6.0,-11.0,B13,Caro-Kann Defense: Exchange Variation,NaN,NaN,2025-01-01 00:00:05
4,aQTAJkjw,NaN,b'\xb3\xc9\xf7\xbf\xf4\x11\xde\x9e\x85*\x97\xc...,"[180, 179, 179, 177, 177, 177, 176, 176, 176, ...","[180, 178, 177, 177, 177, 175, 173, 172, 171, ...",None,112,elprimo27,knocikIII,2139,...,0.0,0-1,Normal,-6.0,5.0,B10,Caro-Kann Defense: Endgame Variation,NaN,NaN,2025-01-01 00:00:05


In [147]:
# Because movedata is compressed binary, accessible through a duckdb extension, we'll need to load it.
# We can extract board positions and generate data from there.

import duckdb
duckdb.sql("LOAD aixchess")

In [148]:
# Let's write some SQL to extract a random board position


In [149]:
# Select out the features and labels
labels = 'result'
features = ['white_rating', 'black_rating']

X_df = df[features]
y_df = df[labels]
X_df, y_df

(       white_rating  black_rating
 0              1247          1218
 1              1577          1593
 2              1043          1000
 3              2015          2028
 4              2139          2145
 ...             ...           ...
 99995          1269           892
 99996          1399          1049
 99997          1407          1380
 99998          2035          2045
 99999          2207          2174
 
 [100000 rows x 2 columns],
 0        1-0
 1        0-1
 2        1-0
 3        1-0
 4        0-1
         ... 
 99995    1-0
 99996    1-0
 99997    1-0
 99998    1-0
 99999    1-0
 Name: result, Length: 100000, dtype: str)

In [150]:
X, y = X_df.to_numpy(), y_df.to_numpy()

In [151]:
# split up into train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [152]:
model = LogisticRegression()

In [153]:
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [154]:
# Let's increase the max iterations and retry.
model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [155]:
y_preds = model.predict(X_test)
y_preds

array(['1-0', '0-1', '0-1', ..., '1-0', '1-0', '1-0'],
      shape=(20000,), dtype=object)

In [156]:
num_correct = (y_preds == y_test).sum()
total = y_test.shape[0]
print(f"Accuracy: {num_correct}/{total} = {(num_correct / total):.3f}")

Accuracy: 10699/20000 = 0.535


In [157]:
# Just barely better than chance. Actually, because of draws, it's not simply 50/50. Let's see the label distribution.
pd.Series(y_train).value_counts()

1-0        39915
0-1        36972
1/2-1/2     3113
Name: count, dtype: int64

In [158]:
# As a percentage:
pd.Series(y_train).value_counts() / y_train.shape[0]

1-0        0.498937
0-1        0.462150
1/2-1/2    0.038913
Name: count, dtype: float64

In [159]:
# Side note: We'll want to filter out the incomplete games (*), even though they represent a small fraction. This is because otherwise we're using 
# up parameters modelling a class that almost never occurs.

In [160]:
# Not a very impressive model.
# Let's do a couple of small modifications, such as normalizing the elo scores beforehand.

print(f"mean white (rating, stddev): {(X_df['white_rating'].mean(), X_df['white_rating'].std())}")

def process_elo(ratings: pd.Series):
    return (ratings - ratings.mean()) / ratings.std()

_X_norm = pd.DataFrame({col: process_elo(X_df[col]) for col in X_df.columns})
print(f"mean white (rating, stddev): {(_X_norm['white_rating'].mean(), _X_norm['white_rating'].std())}")

mean white (rating, stddev): (np.float64(1668.86635), np.float64(398.6814958798929))
mean white (rating, stddev): (np.float64(-2.842170943040401e-17), np.float64(1.0))


In [161]:
# Looks right. Let's create a reusable data processing function, that returns our numpy arrays ready for training.

def preprocess(X_df, y_df, test_size=0.2, random_state=None):
    X_df = pd.DataFrame({col: process_elo(X_df[col]) for col in X_df.columns})
    X, y = X_df.to_numpy(), y_df.to_numpy()
    
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

# Let's create some for training and evaluating a baseline, too.

def train_baseline(X_train, y_train, max_iter=10000) -> LogisticRegression:
    model = LogisticRegression(max_iter=max_iter)
    model.fit(X_train, y_train)
    return model

def eval_model(model, X_test, y_test):
    y_preds = model.predict(X_test)
    num_correct = (y_preds == y_test).sum()
    total = y_test.shape[0]
    print(f"Accuracy: {num_correct}/{total} = {(num_correct / total):.3f}")
    return dict(num_correct=num_correct, total=total, accuracy=num_correct / total)


In [162]:
X_train, X_test, y_train, y_test = preprocess(X_df, y_df, random_state=0)
model = train_baseline(X_train, y_train)
_ = eval_model(model, X_test, y_test)

Accuracy: 10698/20000 = 0.535


In [163]:
# It's actually identical -- normalizaion doesn't do anything. 
# I guess that makes sense, because there's no activation, and it's probably doing a direct solve.
# Let's try changing the random seed, and seeing what some typical scores are.
def benchmark(seeds=range(10)):
    for seed in seeds:
        X_train, X_test, y_train, y_test = preprocess(X_df, y_df, random_state=seed)
        model = train_baseline(X_train, y_train)
        print(f"(seed = {seed})", end=' ')
        _ = eval_model(model, X_test, y_test)

benchmark()

(seed = 0) Accuracy: 10698/20000 = 0.535
(seed = 1) Accuracy: 10752/20000 = 0.538
(seed = 2) Accuracy: 10768/20000 = 0.538
(seed = 3) Accuracy: 10579/20000 = 0.529
(seed = 4) Accuracy: 10699/20000 = 0.535
(seed = 5) Accuracy: 10580/20000 = 0.529
(seed = 6) Accuracy: 10597/20000 = 0.530
(seed = 7) Accuracy: 10726/20000 = 0.536
(seed = 8) Accuracy: 10728/20000 = 0.536
(seed = 9) Accuracy: 10628/20000 = 0.531


In [164]:
# Lucky fluke? Let's check some more.
benchmark(range(50))

(seed = 0) Accuracy: 10698/20000 = 0.535
(seed = 1) Accuracy: 10752/20000 = 0.538
(seed = 2) Accuracy: 10768/20000 = 0.538
(seed = 3) Accuracy: 10579/20000 = 0.529
(seed = 4) Accuracy: 10699/20000 = 0.535
(seed = 5) Accuracy: 10580/20000 = 0.529
(seed = 6) Accuracy: 10597/20000 = 0.530
(seed = 7) Accuracy: 10726/20000 = 0.536
(seed = 8) Accuracy: 10728/20000 = 0.536
(seed = 9) Accuracy: 10628/20000 = 0.531
(seed = 10) Accuracy: 10598/20000 = 0.530
(seed = 11) Accuracy: 10660/20000 = 0.533
(seed = 12) Accuracy: 10621/20000 = 0.531
(seed = 13) Accuracy: 10676/20000 = 0.534
(seed = 14) Accuracy: 10655/20000 = 0.533
(seed = 15) Accuracy: 10614/20000 = 0.531
(seed = 16) Accuracy: 10636/20000 = 0.532
(seed = 17) Accuracy: 10616/20000 = 0.531
(seed = 18) Accuracy: 10583/20000 = 0.529
(seed = 19) Accuracy: 10629/20000 = 0.531
(seed = 20) Accuracy: 10615/20000 = 0.531
(seed = 21) Accuracy: 10692/20000 = 0.535
(seed = 22) Accuracy: 10580/20000 = 0.529
(seed = 23) Accuracy: 10650/20000 = 0.532
(s

In [165]:
# Our first was the best. Strange. Anyway.
# Let's try a heuristic function:
class HeuristicModel():
    def predict(self, X):
        outcomes = np.array(['1-0', '0-1'])

        return outcomes[np.argmax(X + np.array([0.04, 0]), axis=1)]

_ = eval_model(HeuristicModel(), X_test, y_test)

Accuracy: 10720/20000 = 0.536


So the heuristic function is about the same as our LogisticRegression model. 


For the next baseline, we'll actually extract features from the position itself. `03_material_baseline.ipynb` extracts material count and trains a second baseline.